In [1]:
# realizando todos os imports necessários para execução do pipeline
import os
import json
import pickle
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
load_dotenv()

True

In [3]:
# lendo as variáveis de ambiente para definir os caminhos dos diretórios principais
data_path = os.getenv("DATA_PATH")
models_path = os.getenv("MODELS_PATH")
results_path = os.getenv("RESULTS_PATH")

In [4]:
# importando pacote para realizar chamadas no sistema operacional
import subprocess


# criando função que checa se o comando nvidia-smi funciona, indicando presença de gpu
def cuda_available():
    try:
        subprocess.check_output(["nvidia-smi"])
        return True
    except Exception:
        return False


# setando a variável que será repassada aos modelos para garantir uso do hardware correto
DEVICE = "cuda" if cuda_available() else "cpu"
print(f"dispositivo: {DEVICE}")

dispositivo: cpu


In [5]:
# carregando o dataset com os dados que vão alimentar o modelo 2
df = pd.read_parquet(os.path.join(data_path, "model2_dataset.parquet"))
df.head()

,player_id,transfer_date,transfer_fee,predicted_value_m1,buyer_ratio,seller_ratio,buyer_league_tier,seller_league_tier,same_confederation,transfer_window,...,national_team_ranking_inv,goals_per_90,assists_per_90,buyer_net_transfer_record,buyer_national_team_players,buyer_stadium_seats,seller_net_transfer_record,seller_national_team_players,seller_stadium_seats,log_transfer_fee
0,81796,2010-01-01,2400000.0,1299041.375,1.000000,1.000000,4.0,5.0,0,1,...,0.0,0.0,0.0,14570000.0,15.0,60411.0,1260000.0,3.0,66704.0,14.690980
1,66515,2010-01-01,6000000.0,1648851.125,1.000000,1.000000,4.0,4.0,0,1,...,0.0,0.0,0.0,10000000.0,11.0,34725.0,23850000.0,5.0,24584.0,15.607270
2,42457,2010-01-01,600000.0,3445158.250,1.000000,1.000000,4.0,4.0,0,1,...,0.0,0.0,0.0,605000.0,1.0,28678.0,280000.0,11.0,55634.0,13.304687
3,33829,2010-01-09,1400000.0,3417337.500,1.847516,1.847516,1.0,1.0,1,1,...,0.0,0.0,0.0,86560000.0,6.0,32384.0,-47850000.0,20.0,25486.0,14.151984
4,38746,2010-01-11,800000.0,768312.625,1.187063,1.187063,4.0,1.0,1,1,...,0.0,0.0,0.0,4790000.0,2.0,6108.0,-182900000.0,19.0,62850.0,13.592368


In [6]:
# listando as colunas que não devem ser usadas como variável explicativa no treinamento
NON_FEATURES = [
    "player_id",
    "buyer_club_id",
    "seller_club_id",
    "transfer_date",
    "transfer_fee",
    "log_transfer_fee",
]

# filtrando o dataset para manter apenas as colunas que são de fato features
FEATURES = [c for c in df.columns if c not in NON_FEATURES]
FEATURES

['predicted_value_m1',
 'buyer_ratio',
 'seller_ratio',
 'buyer_league_tier',
 'seller_league_tier',
 'same_confederation',
 'transfer_window',
 'transfer_year',
 'player_age_at_transfer',
 'position_rank',
 'national_team_ranking_inv',
 'goals_per_90',
 'assists_per_90',
 'buyer_net_transfer_record',
 'buyer_national_team_players',
 'buyer_stadium_seats',
 'seller_net_transfer_record',
 'seller_national_team_players',
 'seller_stadium_seats']

In [7]:
# ordenando cronologicamente os dados de transferência para evitar vazamento futuro
df = df.sort_values("transfer_date").reset_index(drop=True)

# separando os dados em bases de treino e teste através de split temporal
df_train = df[df["transfer_date"] < "2022-01-01"].copy()
df_test = df[df["transfer_date"] >= "2022-01-01"].copy()

# criando as variáveis explicativas (X) e o target em log (y) para ambas as bases
X_train, y_train = df_train[FEATURES].astype(float), df_train["log_transfer_fee"]
X_test, y_test = df_test[FEATURES].astype(float), df_test["log_transfer_fee"]


In [8]:
# inicializando o gerador de k-folds com k=5 para orientar a validação do optuna
kf = TimeSeriesSplit(n_splits=5)


# declarando função objetivo que treina o modelo iterativamente para o optuna
def objective(trial, quantile_alpha):
    # buscando os melhores parametros
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 300, 1200),
        max_depth=trial.suggest_int("max_depth", 4, 7),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        min_child_weight=trial.suggest_int("min_child_weight", 5, 20),
        gamma=trial.suggest_float("gamma", 0.0, 2.0),
        objective="reg:quantileerror",
        quantile_alpha=quantile_alpha,
        random_state=42,
        tree_method="hist",
        device=DEVICE,
        n_jobs=-1,
    )

    # declarando lista para acumular a pinball loss média em cada iteração do fold
    pinball_folds = []

    # iterando sobre cada partição separando índices de treino e validação gerados pelo kfold
    for tr_idx, cv_idx in kf.split(X_train):
        # dividindo os dataframes de treino e validação para este fold específico
        X_tr, X_cv = X_train.iloc[tr_idx], X_train.iloc[cv_idx]
        y_tr, y_cv = y_train.iloc[tr_idx], y_train.iloc[cv_idx]

        # instanciando o modelo usando os parâmetros sugeridos com parada antecipada
        m = XGBRegressor(**params, early_stopping_rounds=30)

        # executando treinamento do modelo na base de treino provisória
        m.fit(X_tr, y_tr, eval_set=[(X_cv, y_cv)], verbose=False)

        # calculando a pinball loss no validation fold
        preds = m.predict(X_cv)
        errors = y_cv.values - preds
        loss = np.maximum(quantile_alpha * errors, (quantile_alpha - 1) * errors)
        pinball_folds.append(np.mean(loss))

    # retornando a média do erro avaliado nos 5 folds para ser minimizada pelo optuna
    return float(np.mean(pinball_folds))

In [9]:
# criando e executando os estudos do optuna focando em minimizar a pinball loss de cada quantil separadamente
study_p15 = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_p15.optimize(lambda trial: objective(trial, 0.15), n_trials=50)
best_params_p15 = study_p15.best_params
best_params_p15.update({"objective": "reg:quantileerror", "tree_method": "hist", "device": DEVICE, "random_state": 42, "n_jobs": -1})

study_p50 = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_p50.optimize(lambda trial: objective(trial, 0.50), n_trials=50)
best_params_p50 = study_p50.best_params
best_params_p50.update({"objective": "reg:quantileerror", "tree_method": "hist", "device": DEVICE, "random_state": 42, "n_jobs": -1})

study_p85 = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_p85.optimize(lambda trial: objective(trial, 0.85), n_trials=50)
best_params_p85 = study_p85.best_params
best_params_p85.update({"objective": "reg:quantileerror", "tree_method": "hist", "device": DEVICE, "random_state": 42, "n_jobs": -1})

print("Melhores parâmetros P50 encontrados:", best_params_p50)

[I 2026-07-21 01:16:43,389] A new study created in memory with name: no-name-2324214a-ab1a-4bb7-a624-d69c7d6a9231
[I 2026-07-21 01:16:46,423] Trial 0 finished with value: 0.15782490584110057 and parameters: {'n_estimators': 637, 'max_depth': 7, 'learning_rate': 0.07259248719561363, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.6624074561769746, 'reg_alpha': 0.029375384576328288, 'reg_lambda': 0.13066739238053282, 'min_child_weight': 18, 'gamma': 1.2022300234864176}. Best is trial 0 with value: 0.15782490584110057.
[I 2026-07-21 01:16:47,710] Trial 1 finished with value: 0.15648600596297038 and parameters: {'n_estimators': 937, 'max_depth': 4, 'learning_rate': 0.13826189316223852, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.035113563139704075, 'reg_lambda': 0.2327067708383781, 'min_child_weight': 9, 'gamma': 1.0495128632644757}. Best is trial 1 with value: 0.15648600596297038.
[I 2026-07-21 01:16:48,892] Trial 2 finished with value: 0

Melhores parâmetros P50 encontrados: {'n_estimators': 1028, 'max_depth': 5, 'learning_rate': 0.012865313502376243, 'subsample': 0.7186730579651694, 'colsample_bytree': 0.6021186734226314, 'reg_alpha': 0.8925979401647732, 'reg_lambda': 0.59641556808375, 'min_child_weight': 7, 'gamma': 0.9584833985899621, 'objective': 'reg:quantileerror', 'tree_method': 'hist', 'device': 'cpu', 'random_state': 42, 'n_jobs': -1}


In [10]:
# instanciando os modelos de regressão quantílica utilizando seus hiperparâmetros específicos
model_p15 = XGBRegressor(**best_params_p15, quantile_alpha=0.15)
model_p50 = XGBRegressor(**best_params_p50, quantile_alpha=0.50)
model_p85 = XGBRegressor(**best_params_p85, quantile_alpha=0.85)

model_p15.fit(X_train, y_train)
model_p50.fit(X_train, y_train)
model_p85.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7390907986673838, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.30579255416008805,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05033120721874637,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=10, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=325, n_jobs=-1,
             num_parallel_tree=None, objective='reg:quantileerror', ...)

In [11]:
# executando o ajuste/treinamento dos 3 modelos paralelamente na base total de treinamento
model_p15.fit(X_train, y_train)
model_p50.fit(X_train, y_train)
model_p85.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7390907986673838, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.30579255416008805,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05033120721874637,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=10, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=325, n_jobs=-1,
             num_parallel_tree=None, objective='reg:quantileerror', ...)

In [12]:
# registrando em formato pickle a lista com os nomes exatos das features utilizadas na modelagem
with open(os.path.join(models_path, "feature_names_model2.pkl"), "wb") as f:
    pickle.dump(FEATURES, f)

In [13]:
# gravando em disco o primeiro modelo para projeção do limite inferior de preço (p15)
model_p15.save_model(os.path.join(models_path, "model2_transfer_fee_p15.json"))

# gravando em disco o segundo modelo central com base no comportamento mediano (p50)
model_p50.save_model(os.path.join(models_path, "model2_transfer_fee_p50.json"))

# gravando em disco o terceiro e último modelo que define o teto do intervalo estimado (p85)
model_p85.save_model(os.path.join(models_path, "model2_transfer_fee_p85.json"))

In [14]:
# avaliando a mediana no conjunto de testes
pred_p50 = model_p50.predict(X_test)

y_test_real = np.expm1(y_test.values)
pred_real = np.expm1(pred_p50)

mae_final = mean_absolute_error(y_test_real, pred_real)
mape_final = np.mean(np.abs((y_test_real - pred_real) / (y_test_real + 1e-6))) * 100
rmse_final = np.sqrt(mean_squared_error(y_test_real, pred_real))
rmse_log = np.sqrt(mean_squared_error(y_test, pred_p50))
r2_final_log = r2_score(y_test, pred_p50)
r2_final_real = r2_score(y_test_real, pred_real)

print(f"MAE: € {mae_final:>12,.0f}")
print(f"MAPE: {mape_final:>12.2f} %")
print(f"RMSE (€): € {rmse_final:>12,.0f}")
print(f"RMSE (log): {rmse_log:>12.4f}")
print(f"R² (log): {r2_final_log:>12.4f}")
print(f"R² (real): {r2_final_real:>11.4f}")

MAE: €    3,747,277
MAPE:        56.85 %
RMSE (€): €    7,777,266
RMSE (log):       0.6629
R² (log):       0.6730
R² (real):      0.5995


In [15]:
# salvando as métricas do modelo 2 (P50)
metrics = {
    "dataset": "validation",
    "split": {
        "train_end": "2021-12-31",
        "val_start": "2022-01-01",
    },
    "n_train": int(len(X_train)),
    "n_val": int(len(X_test)),
    "n_features": int(len(FEATURES)),
    "device": DEVICE,
    "MAE_eur": float(mae_final),
    "MAPE_pct": float(mape_final),
    "RMSE_eur": float(rmse_final),
    "RMSE_log": float(rmse_log),
    "R2_log": float(r2_final_log),
    "R2_real": float(r2_final_real),
    "best_params_p50": best_params_p50,
    "optuna_trials": 50,
    "optuna_best_pinball_p50": float(study_p50.best_value),
}

out_path = os.path.join(results_path, "metrics_model2_train.json")
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)